# Discrete Fourier Analysis

Implement the DFT and IDFT directly, compare with NumPy, and examine sampling and spectral resolution.

Run the cells in order. This notebook is independent of the other notebooks.


In [ ]:
from pathlib import Path

# Resolve paths when launched from the repository root or notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "audio"
OUTPUT_DIR = ROOT / "outputs"
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Direct DFT and IDFT


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple

# ------------------------- DFT / IDFT Utilities ------------------------------

def _dft_vandermonde(N: int, sign: int = -1) -> np.ndarray:
    """
    Build the N×N complex exponential matrix:
      W[k,n] = exp(j * sign * 2π kn / N)
    sign = -1 -> DFT, sign = +1 -> IDFT kernel.
    """
    n = np.arange(N)
    k = n.reshape(N, 1)
    return np.exp(1j * sign * 2 * np.pi * k * n / N)

def dft_naive(x: np.ndarray) -> np.ndarray:
    """O(N^2) DFT (matrix multiply), returns complex spectrum."""
    x = np.asarray(x, dtype=complex)
    W = _dft_vandermonde(x.size, sign=-1)
    return W @ x

def idft_naive(X: np.ndarray) -> np.ndarray:
    """O(N^2) IDFT with 1/N normalization (matrix multiply)."""
    X = np.asarray(X, dtype=complex)
    N = X.size
    W = _dft_vandermonde(N, sign=+1)
    return (W @ X) / N


# --------------------------- Signal Construction -----------------------------

def build_cos_pi_tone(T: float, N: int) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Construct x[n] = cos(pi * n * T) with indices centered around zero.
    Returns (n_values, x, fs).
    """
    fs = 1.0 / T
    n_vals = np.arange(-N // 2, -N // 2 + N, dtype=float)
    x = np.cos(np.pi * (n_vals * T))
    return n_vals, x.astype(float), fs

def plot_window(sig: np.ndarray, start: int = 450, stop: int = 550, title: str = "") -> None:
    """Replicates the original 'windowed view' plot with the same slice and styling."""
    sl = slice(start, stop)
    idx = np.arange(sig.size)[sl]
    plt.figure(figsize=(8, 3))
    plt.plot(idx, sig[sl], color="red")
    plt.title(title or r"Windowed view of $x[n]=\cos(\pi n T)$")
    plt.xlabel("sample index")
    plt.ylabel("amplitude")
    plt.tight_layout()
    plt.show()

# ------------------------------- Experiment ----------------------------------

T = 1 / 2
N = 1000
n_vals, x = None, None

n_vals, x, fs = build_cos_pi_tone(T=T, N=N)
assert len(n_vals) == N

x_idx = x.copy()

plot_window(
    x_idx, 450, 550,
    title="x[n] = cos(pi n T),  T=1/2,  N=1000"
)

## Comparison with NumPy FFT


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- Helpers ----------
def freq_axis_hz(N: int, fs: float) -> np.ndarray:
    """Frequency axis in Hz for DFT bins 0..N-1."""
    return np.arange(N) * fs / N

def plot_stem_mag(x_axis: np.ndarray,
                  X: np.ndarray,
                  title: str,
                  xlabel: str,
                  xlim: tuple | None = None) -> None:
    """Stem-plot of |X| with consistent styling."""
    plt.figure(figsize=(8, 3))
    plt.stem(x_axis, np.abs(X), basefmt=" ", linefmt='green')
    if xlim is not None:
        plt.xlim(*xlim)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("|X[k]|")
    plt.tight_layout()
    plt.show()

def dft_vs_numpy_plots(x_idx: np.ndarray, fs: float) -> None:
    """Compute DFTs and reproduce the two diagnostic plots + max|diff| print."""
    N = x_idx.size
    k = np.arange(N)
    freq = freq_axis_hz(N, fs)

    X_ref = dft_naive(x_idx)
    X_np  = np.fft.fft(x_idx)

    print("Maximum difference between the numpy FFT and implemented DFT:", np.max(np.abs(X_ref - X_np)))
    print("\n")


    # Plot 1: |X[k]| vs frequency (Hz)
    plot_stem_mag(
        freq, X_ref,
        title="DFT magnitude (peaks at 0.5 Hz and wrapped -0.5 Hz)",
        xlabel="frequency (Hz)",
        xlim=(0, fs),
    )

    print("\n")

    # Plot 2: zoom around expected bins (k=250 and k=750)
    plot_stem_mag(
        k, X_ref,
        title="DFT magnitude -> k=750 and k=250",
        xlabel="index k",
        xlim=(200, 800)
    )

    return X_ref, X_np, freq

X_ref, X_np, freq = dft_vs_numpy_plots(x_idx, fs)

## Time-domain reconstruction


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable, Tuple

def reconstruct_time_domain(X: np.ndarray,
                            ifft_fn: Callable[[np.ndarray], np.ndarray]) -> np.ndarray:
    """Apply IDFT to spectrum X using the provided inverse function."""
    return ifft_fn(X)

def compare_original_vs_recon(x_orig: np.ndarray,
                              x_recon: np.ndarray,
                              title: str = "Time-domain reconstruction with IDFT") -> float:
    """
    Plot original vs reconstructed signals and return the max absolute error.
    """
    err = np.max(np.abs(x_recon - x_orig))
    print("Maximum reconstruction error:", err)

    plt.figure(figsize=(9, 3))
    plt.plot(x_orig, label="Org", alpha=0.8, color = "green")
    plt.plot(x_recon.real, "--", label="Reconstructed", alpha=0.8, color = "red")
    plt.title(title)
    plt.xlabel("sample index")
    plt.ylabel("amplitude")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    return err

x_rec = reconstruct_time_domain(X_ref, idft_naive)
_ = compare_original_vs_recon(x_idx, x_rec, title="Reconstruction of time-domain using IDFT")

## Sampling and spectral resolution


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import ceil
from typing import Iterable, Tuple, List


def generate_centered_cosine(num_samples: int, Ts: float) -> Tuple[np.ndarray, float]:
    """
    Create x[n] = cos(pi * n * Ts) with n centered around 0.
    Returns (signal, fs).
    """
    fs = 1.0 / Ts
    n0 = -num_samples // 2
    n = np.arange(n0, n0 + num_samples, dtype=float)
    x = np.cos(np.pi * n * Ts)
    return x, fs


def dft_with_freq_axis(x: np.ndarray, fs: float) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute the DFT X[k] and its frequency axis f[k] in Hz for 0..N-1.
    """
    N = x.size
    X = np.fft.fft(x)
    k = np.arange(N)
    f = k * fs / N
    return X, f


def _grid_shape(nplots: int) -> Tuple[int, int]:
    """
    Simple grid: 2 columns, enough rows for all plots.
    """
    cols = 2
    rows = ceil(nplots / cols)
    return rows, cols


class SpectrumPlotter:
    """Responsible only for plotting spectra."""

    def __init__(self, suptitle: str = "DFT magnitude for different N and Ts"):
        self.suptitle = suptitle

    def plot_family(
        self,
        spectra: List[Tuple[np.ndarray, np.ndarray, int, float]],
        *,
        xlim_to_fs: bool = True
    ) -> None:
        """
        Plot multiple |X[k]| vs f[k].
        spectra: list of tuples (X, f, N, Ts)
        """
        nplots = len(spectra)
        rows, cols = _grid_shape(nplots)
        fig = plt.figure(figsize=(10, 6.5))
        for i, (X, f, N, Ts) in enumerate(spectra, start=1):
            ax = fig.add_subplot(rows, cols, i)
            markerline, stemlines, baseline = ax.stem(f, np.abs(X), basefmt=" ", linefmt='red')
            ax.set_title(f"T={Ts:g} s, fs={1.0/Ts:g} Hz, N={N}")
            ax.set_xlabel("Hz")
            ax.set_ylabel("|X[k]|")
            if xlim_to_fs:
                ax.set_xlim(0, 1.0 / Ts)
            ax.grid(True, alpha=0.4)

        fig.suptitle(self.suptitle)
        fig.tight_layout(rect=[0, 0, 1, 0.9])
        plt.show()


class DFTExperiment:
    """
    Orchestrates:
      - building signals for (N, Ts) pairs,
      - computing DFTs,
      - delegating plotting.
    """

    def __init__(self):
        self.plotter = SpectrumPlotter(
            suptitle="Effect of resolution (N) and sampling rate (T) on DFT of x[n]"
        )

    def run(self, Ns: Iterable[int], Ts: Iterable[float]) -> None:
        """
        For each T in Ts, produce one figure containing all N in Ns.
        """
        Ns = list(Ns)
        Ts = list(Ts)

        for Ts_i in Ts:
            spectra_for_T: List[Tuple[np.ndarray, np.ndarray, int, float]] = []
            for N in Ns:
                x, fs = generate_centered_cosine(N, Ts_i)
                X, f = dft_with_freq_axis(x, fs)
                spectra_for_T.append((X, f, N, Ts_i))

            self.plotter.plot_family(spectra_for_T)


if __name__ == "__main__":
    Ns = (128, 256, 512, 1000)
    Ts_list = (0.5, 0.25, 1.0)
    DFTExperiment().run(Ns, Ts_list)